# 04 · Consumo por unidad consumidora y mercado

Práctica basada en `notebooks/02_energia.ipynb`. Tiempo sugerido: 20–30 minutos.

**Tu tarea:** completá únicamente la función de la sección 3.
La carga de datos, el entorno y las pruebas ya están preparados.

1. Elegí el kernel del entorno `.venv` del proyecto y ejecutá las celdas de preparación.
2. Leé el contrato y reemplazá el `raise NotImplementedError` por tu implementación.
3. Ejecutá de nuevo la celda de la función y después la de pruebas.
4. Si todas pasan, esa misma celda probará tu función sobre los datos reales y mostrará el resultado.

Podés usar **Ejecutar todo** desde un kernel nuevo. Mientras la función esté pendiente,
la última celda mostrará un aviso; no tenés que instalar nada adicional en el entorno actual.

## 1. Preparación lista

La base equivale a `base` del notebook original: fecha como índice, nombres preparados y faltantes, negativos y ceros conservados.

Los archivos de `../datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_energia()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 285,499 | Columnas: 13


## 2. Consigna y resultado esperado

Construí una tabla por mes y `TipoConsumidor` con el consumo por unidad consumidora.

1. Seleccioná primero las filas donde **ambas** mediciones, `Consumo` y `Consumidores`, estén presentes.
2. Agrupá por inicio de mes (`MS`) y mercado, y sumá ambas columnas.
3. Calculá `MWh_por_unidad = Consumo / Consumidores` solamente si el total de
   consumidores del grupo es mayor que cero; en caso contrario, dejá `NaN`.

Devolvé columnas `Consumo`, `Consumidores`, `MWh_por_unidad`, en ese orden,
y un MultiIndex llamado `Data`, `TipoConsumidor`, ordenado por fecha y mercado ascendente.
Incluí solo grupos con registros completos. Conservá consumos y consumidores negativos
al sumar; la condición del denominador se aplica **después** de agrupar.
No calcules la media de cocientes individuales ni modifiques la entrada.

## 3. Tu función

Editá solo esta celda. Mantené el nombre de la función y sus parámetros para que funcionen las pruebas.

In [3]:
def calcular_intensidad_por_mercado(datos):
    """Implementá el contrato de la sección 2 sin modificar datos."""
    # TODO: escribí tu solución acá.
    raise NotImplementedError("Completá esta función y volvé a ejecutar las pruebas.")

## 4. Pruebas y ejecución real

Un `AssertionError` indica qué resultado difiere del esperado. Primero revisá el caso pequeño que falló. Después de editar la función, ejecutá su celda antes de volver a correr estas pruebas.

Las pruebas verifican ejemplos pequeños y propiedades del resultado real; no sustituyen una revisión de tu razonamiento.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "TipoConsumidor": ["Cativo", "Cativo", "Cativo", "Livre", "Livre", "Cativo", "Livre", "Livre"],
        "Consumo": [10, np.nan, 100, 20, -2, 30, -4, 5],
        "Consumidores": [2, 100, np.nan, 0, 0, 3, 2, -1],
    }, index=pd.DatetimeIndex(
        ["2023-01-01"] * 5 + ["2023-02-01"] * 2 + ["2023-03-01"], name="Data",
    ))
    original = caso.copy(deep=True)
    indice = pd.MultiIndex.from_arrays([
        pd.to_datetime(["2023-01-01", "2023-01-01", "2023-02-01", "2023-02-01", "2023-03-01"]),
        ["Cativo", "Livre", "Cativo", "Livre", "Livre"],
    ], names=["Data", "TipoConsumidor"])
    esperado = pd.DataFrame({
        "Consumo": [10, 18, 30, -4, 5],
        "Consumidores": [2, 0, 3, 2, -1],
        "MWh_por_unidad": [5, np.nan, 10, -2, np.nan],
    }, index=indice)
    comparar(funcion(caso), esperado)
    comparar(funcion(caso.iloc[::-1]), esperado)

    # Cociente de sumas: (10 + 90) / (1 + 3) = 25; la media de 10 y 30 sería 20.
    ponderado = pd.DataFrame({
        "TipoConsumidor": ["Cativo", "Cativo"],
        "Consumo": [10, 90], "Consumidores": [1, 3],
    }, index=pd.DatetimeIndex(["2023-01-01"] * 2, name="Data"))
    esperado_ponderado = esperado.iloc[:1].copy()
    esperado_ponderado.iloc[0] = [100, 4, 25]
    comparar(funcion(ponderado), esperado_ponderado)

    comparar(funcion(caso.iloc[:0]), esperado.iloc[:0])
    comparar(caso, original)
    print("OK: faltantes emparejados, cociente de sumas, negativos, denominadores y orden.")


resultado = None
try:
    probar(calcular_intensidad_por_mercado)
except NotImplementedError:
    print("PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.")
else:
    # Comprobamos también que el dataset real permanezca intacto.
    antes = datos.copy(deep=True)
    resultado = calcular_intensidad_por_mercado(datos)
    comparar(datos, antes)
    completos = datos.dropna(subset=["Consumo", "Consumidores"])
    assert resultado.index.names == ["Data", "TipoConsumidor"]
    assert resultado.index.is_monotonic_increasing
    np.testing.assert_allclose(resultado["Consumo"].sum(), completos["Consumo"].sum())
    np.testing.assert_allclose(resultado["Consumidores"].sum(), completos["Consumidores"].sum())
    assert resultado.loc[resultado["Consumidores"].le(0), "MWh_por_unidad"].isna().all()
    print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
    display(resultado.head(20))

PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.


## Pista opcional

<details>
<summary>Abrir solo si te trabás</summary>

`dropna(subset=...)`, `pd.Grouper(freq='MS')`, `groupby` y `where` para el denominador.

</details>

**Para explicar en voz alta:** ¿qué filas entran al cálculo?, ¿qué representa cada fila de la salida?, ¿qué caso límite cubre cada prueba?